<a href="https://colab.research.google.com/github/elfrontman/assesment3_is/blob/heidy-hf-preprocessing/notebooks/02_preprocessing_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ISY503 — Assessment 3: Data Preprocessing & Hugging Face Tokenization

**Project:** NLP Sentiment Analysis on Amazon Product Reviews  
**Notebook:** 02_preprocessing_huggingface.ipynb  
**Branch:** heidy-hf-preprocessing  
**Owner:** Heidy

## Objective

This notebook covers Steps 4–6 of the team workflow: Train / Validation / Test splitting, Hugging Face GPT-2 tokenization, padding and truncation, and dataset preparation for model training.

The objective is to transform the cleaned Amazon product reviews into structured inputs that can be directly used during GPT-2 fine-tuning.

**Core question:**

How can we prepare the Amazon reviews for GPT-2 while preserving sentiment labels, preventing data leakage, and ensuring compatibility with Hugging Face?


## 0. Install Required Libraries

Install the Python libraries required for data preprocessing, GPT-2 tokenization and Hugging Face dataset preparation.

These libraries will also support the next stage of the project, including model training and fine-tuning.

In [8]:
# Install the libraries required for preprocessing and GPT-2

%pip install -q transformers datasets scikit-learn





## 1. Setup & Configuration

Import the required libraries and configure reproducibility, file paths and project settings.

Google Colab will be used as the development environment. The repository will be accessed through GitHub.

In [9]:

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import GPT2TokenizerFast
from datasets import Dataset, DatasetDict

from sklearn.model_selection import train_test_split

# Reproducibility
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# Project configuration
MODEL_NAME = "gpt2"

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

# Repository paths
REPO_ROOT = Path("/content/assesment3_is")

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR = REPO_ROOT / "outputs" / "preprocessing"

SPLIT_DIR = OUTPUT_DIR / "splits"
TOKENIZED_DIR = OUTPUT_DIR / "tokenized_dataset"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = PROCESSED_DIR / "amazon_reviews_no_duplicates.csv"

print(f"Repository root: {REPO_ROOT}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Repository root: /content/ISY503
Processed data: /content/ISY503/data/processed
Output directory: /content/ISY503/outputs/preprocessing
Model: gpt2
PyTorch version: 2.14.1+cpu
CUDA available: False



### 1.2 Environment Verification

Before proceeding with the preprocessing workflow, we verify that the required libraries have been imported successfully.

Checking the installed versions helps maintain reproducibility and identify potential compatibility issues during development.


In [10]:

# Check current Google Colab location

import os

print(os.getcwd())


/content
